# Lab 5: A generative student with LoRA

The student is now a model that writes text. It is trained to write the type of each message,
first with every weight moving, then with LoRA.

You will:

1. Fully fine-tune the 0.5B student.
2. Train the same student with LoRA, and compare cost and accuracy.
3. Merge the adapter and check that nothing changed.
4. Train a 1.5B student with LoRA.
5. Compare two ranks on validation data.

This lab needs results from Labs 2 and 3. Allow about 40 minutes. Run the cells in order.

In [ ]:
from kd_course.env import check_environment
from kd_course.results import default_results_dir, require_result

report = check_environment()
print(report.summary())
assert report.ok, "Fix the problems listed above before continuing."

baseline_result = require_result("baseline_gold_616")
scores_path = default_results_dir() / "teacher_scores" / "pool.json"
assert scores_path.is_file(), "Run Lab 3 first: this lab uses the teacher scores it saves."

## 1. The data

The same for every run: the gold messages with their human labels, plus the pool labelled with the
teacher's top choice. This is distillation; LoRA only changes how the student's weights are updated.

In [ ]:
from kd_course.data import LabelledSet, load_banking77, make_splits
from kd_course.label_scoring import load_scores, top_labels

bundle = load_banking77()
splits = make_splits(bundle.train, bundle.test)
LABELS = bundle.label_names
N = len(LABELS)

pool_labels = top_labels(load_scores(scores_path, splits.pool.texts))
train = LabelledSet(
    texts=splits.gold_train.texts + splits.pool.texts,
    labels=splits.gold_train.labels + pool_labels,
)
print(f"{len(train)} training messages")

## 2. One function for every run

`rank=None` means full fine-tuning. Any other value attaches LoRA adapters of that rank.
The function measures time and peak memory, scores the student, and frees the GPU.

In [ ]:
import gc

import torch

from kd_course.generative import measure_training, predict_generative, train_generative_classifier
from kd_course.lora import add_lora, count_parameters_by_kind, folder_size_mb
from kd_course.metrics import accuracy, summarize
from kd_course.models import load_causal_lm
from kd_course.results import save_result

costs = {}
predictions = {}
val_scores = {}


def run(name, model_id, rank=None, save_test_result=True):
    lora = rank is not None
    model, tokenizer = load_causal_lm(model_id, dtype="bfloat16" if lora else "float32")
    if lora:
        model = add_lora(model, rank=rank, alpha=2 * rank)
    counts = count_parameters_by_kind(model)

    with measure_training(model) as measured:
        train_generative_classifier(
            model, tokenizer, train, LABELS, epochs=2, learning_rate=2e-4 if lora else 2e-5
        )
    costs[name] = {
        "trained share": f"{counts['trainable_share']:.2%}",
        "peak GB": round(measured.cost.peak_memory_gb, 1),
        "minutes": round(measured.cost.seconds / 60, 1),
    }

    val_pred = predict_generative(model, tokenizer, splits.gold_val.texts, LABELS)
    val_scores[name] = accuracy(splits.gold_val.labels, val_pred)
    if save_test_result:
        predictions[name] = predict_generative(model, tokenizer, splits.test.texts, LABELS)
        save_result(name, summarize(splits.test.labels, predictions[name], N), predictions[name])
    print(name, costs[name], f"validation accuracy {val_scores[name]:.1%}")
    return model, tokenizer


def free():
    """Release GPU memory. Delete your own references to a model before calling this."""
    gc.collect()
    torch.cuda.empty_cache()

## 3. Run A: the 0.5B student, fully fine-tuned

In [ ]:
from kd_course.lora import STUDENT_MEDIUM_ID, STUDENT_SMALL_ID

folder = default_results_dir() / "students"

model, tokenizer = run("generative_05B_full", STUDENT_SMALL_ID)
model.save_pretrained(folder / "05B_full")
del model
free()

## 4. Run B: the same student with LoRA

The adapter is saved on its own. Compare its size on disk with the full model from run A.

In [ ]:
lora_model, tokenizer = run("generative_05B_lora_r16", STUDENT_SMALL_ID, rank=16)
lora_model.save_pretrained(folder / "05B_lora_r16")

print(f"Full model on disk: {folder_size_mb(folder / '05B_full'):,.0f} MB")
print(f"Adapter on disk:    {folder_size_mb(folder / '05B_lora_r16'):,.1f} MB")

## 5. Merge the adapter and check

The merged model should give the same answers as the unmerged one.

In [ ]:
from kd_course.lora import merge_lora

sample = splits.test.texts[:200]
lora_model.eval()
before = predict_generative(lora_model, tokenizer, sample, LABELS)
merged = merge_lora(lora_model)
after = predict_generative(merged, tokenizer, sample, LABELS)

same = sum(a == b for a, b in zip(before, after, strict=True))
print(f"{same} of {len(sample)} answers are the same after merging.")

del lora_model, merged
free()

## 6. Run C: a 1.5B student with LoRA

Three times larger, on the same card.

In [ ]:
model, tokenizer = run("generative_15B_lora_r16", STUDENT_MEDIUM_ID, rank=16)
del model
free()

## 7. Does rank matter here?

Two more runs on the 0.5B student, compared on the **validation** set only. Their test scores
are not computed, so they cannot tempt you.

In [ ]:
for rank in (4, 64):
    model, tokenizer = run(f"rank_{rank}", STUDENT_SMALL_ID, rank=rank, save_test_result=False)
    del model
    free()

for name in ("rank_4", "generative_05B_lora_r16", "rank_64"):
    print(f"{name:<26} validation accuracy {val_scores[name]:.1%}   {costs[name]}")

## 8. Compare

In [ ]:
from kd_course.metrics import paired_bootstrap
from kd_course.results import format_results, load_results

print("Cost of each run")
for name, cost in costs.items():
    print(f"  {name:<26} {cost}")
print()
print(format_results(load_results()))


def report(title, a, b):
    r = paired_bootstrap(accuracy, splits.test.labels, a, b)
    print(
        f"{title}: {r.difference * 100:+.1f} points "
        f"(95% interval {r.low * 100:+.1f} to {r.high * 100:+.1f})"
    )


print()
report(
    "LoRA minus full fine-tune (0.5B)",
    predictions["generative_05B_full"],
    predictions["generative_05B_lora_r16"],
)
report(
    "1.5B LoRA minus 0.5B LoRA       ",
    predictions["generative_05B_lora_r16"],
    predictions["generative_15B_lora_r16"],
)

## 9. Optional: QLoRA on the 7B model

This cell loads the 7B model with 4-bit weights, attaches adapters, and trains on 320 messages,
only to show that it fits. It needs the `bitsandbytes` library and may not work on every setup.
Skip it if it fails.

In [ ]:
from kd_course.models import TEACHER_ID, load_causal_lm_4bit

big, big_tokenizer = load_causal_lm_4bit(TEACHER_ID)
big = add_lora(big, rank=16, alpha=32)
print(count_parameters_by_kind(big))

small_train = LabelledSet(texts=train.texts[:320], labels=train.labels[:320])
with measure_training(big) as measured:
    train_generative_classifier(
        big, big_tokenizer, small_train, LABELS, epochs=1, learning_rate=2e-4, batch_size=16
    )
print(f"Peak GPU memory: {measured.cost.peak_memory_gb:.1f} GB")

## 10. What did you see?

Double-click this cell and write your answers.

**a. What share of the weights did LoRA train on the 0.5B student? How did peak memory and time compare with full fine-tuning?**

**b. Is the accuracy difference between LoRA and full fine-tuning real? Quote the interval.**

**c. How much smaller is the adapter on disk than the full model? When would that matter to you?**

**d. Did the 1.5B student earn its extra cost on this task? What kind of task might change your answer?**

**e. Did rank 4 differ from rank 64 on validation? With 308 validation messages, how sure can you be?**

**f. Compare the generative students with the encoder students from Lab 3. Which would you serve for this task, and why?**